# 🎓 University Final Project: 2D Image to 3D Model AI Backend
### Fast 360° 3D Mesh Reconstruction Server powered by Google Colab GPU (T4)
### Model: `idwnis/TripoSR-bucket`

---
### 📌 Quick Instructions:
1. Ensure you are connected to a **GPU Runtime**: 
   - Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
2. If your repository `idwnis/TripoSR-bucket` is **private**, set your Hugging Face token in Step 1.5.
3. Click **Runtime** -> **Run all**.
4. Cell 4 will output your public HTTPS URL (e.g. `https://xxxx.trycloudflare.com`).
5. Copy and paste that URL into your **3D Vision Studio** Web App settings!

In [ ]:
# Step 1: Verify NVIDIA GPU & Install Dependencies
!nvidia-smi

import os, sys

# 1. Clone TripoSR repo
if not os.path.exists('/content/TripoSR'):
    print("[*] Cloning TripoSR repository...")
    !git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

sys.path.insert(0, '/content/TripoSR')

print("[*] Installing dependencies (PyTorch 3D modules, FastAPI, Cloudflared)...")
# Install only the necessary packages without altering Colab's native NumPy/SciPy
!pip install -q einops omegaconf trimesh transformers fastapi uvicorn python-multipart huggingface_hub
!pip install -q git+https://github.com/tatsy/torchmcubes.git

# Download cloudflared tunnel binary for instant public HTTPS endpoint
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("[+] All packages and tools installed successfully!")

In [ ]:
# Step 1.5: (Optional) Hugging Face Login for Private Repositories
# If 'idwnis/TripoSR-bucket' is private, enter your HF token below or leave empty if public.
import os
HF_TOKEN = ""  # <-- Paste your Hugging Face read token here if the repo is private (hf_...)

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    from huggingface_hub import login
    login(token=HF_TOKEN.strip())
    print("[+] Logged in to Hugging Face successfully!")
else:
    print("[*] No HF token provided. Proceeding with public access.")

In [ ]:
# Step 2: Load TripoSR Model (idwnis/TripoSR-bucket) onto T4 GPU
import os, sys
if os.path.exists('/content/TripoSR') and '/content/TripoSR' not in sys.path:
    sys.path.insert(0, '/content/TripoSR')

import torch
from PIL import Image
from tsr.system import TSR

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing TripoSR on device: {device}...")

PRIMARY_REPO = "idwnis/TripoSR-bucket"
FALLBACK_REPO = "stabilityai/TripoSR"
hf_token = os.environ.get("HF_TOKEN", None)

try:
    print(f"[*] Loading model weights from '{PRIMARY_REPO}'...")
    tsr_model = TSR.from_pretrained(
        PRIMARY_REPO,
        config_name="config.yaml",
        weight_name="model.ckpt",
        token=hf_token
    )
    active_repo = PRIMARY_REPO
    print(f"[+] Successfully loaded model from '{PRIMARY_REPO}'!")
except Exception as e:
    print(f"[!] Notice: Could not load from '{PRIMARY_REPO}': {e}")
    print(f"[*] Falling back to public '{FALLBACK_REPO}'...")
    tsr_model = TSR.from_pretrained(
        FALLBACK_REPO,
        config_name="config.yaml",
        weight_name="model.ckpt",
    )
    active_repo = FALLBACK_REPO
    print(f"[+] Successfully loaded fallback from '{FALLBACK_REPO}'!")

tsr_model.renderer.set_chunk_size(8192)
tsr_model.to(device)

gpu_title = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"[+] Model ({active_repo}) ready on: {gpu_title}!")

In [ ]:
# Step 3: Define FastAPI Endpoints
import io
import time
import numpy as np
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from tsr.utils import resize_foreground

app = FastAPI(title="3D Vision Studio API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def prepare_image(pil_img: Image.Image) -> Image.Image:
    """Prepares image for TripoSR: ensures RGBA format and centers foreground."""
    if pil_img.mode == 'RGBA':
        img_rgba = pil_img
    else:
        img_rgba = pil_img.convert("RGBA")
        # Auto-remove pure white or solid corners if background is uniform
        data = np.array(img_rgba)
        r, g, b = data[:, :, 0], data[:, :, 1], data[:, :, 2]
        white_mask = (r > 240) & (g > 240) & (b > 240)
        data[:, :, 3][white_mask] = 0
        img_rgba = Image.fromarray(data)
    
    return resize_foreground(img_rgba, ratio=0.85)

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "ok",
        "gpu_name": gpu_name,
        "vram_allocated_gb": round(vram_alloc, 2),
        "vram_total_gb": round(vram_total, 2),
        "model_ready": True,
        "model_repo": active_repo,
        "timestamp": time.time()
    }

@app.post("/api/generate")
async def generate_3d(image: UploadFile = File(...)):
    try:
        contents = await image.read()
        pil_img = Image.open(io.BytesIO(contents))
        
        # Prepare image foreground
        foreground = prepare_image(pil_img)
        
        # 3D Neural reconstruction
        with torch.no_grad():
            scene_codes = tsr_model([foreground], device=device)
            meshes = tsr_model.extract_mesh(scene_codes, resolution=256, has_texture=True)
        
        mesh = meshes[0]
        glb_io = io.BytesIO()
        mesh.export(glb_io, file_type="glb")
        
        return Response(
            content=glb_io.getvalue(),
            media_type="model/gltf-binary",
            headers={"Content-Disposition": 'attachment; filename="model.glb"'}
        )
    except Exception as e:
        print(f"[!] Generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

print("[+] FastAPI Application defined.")

In [ ]:
# Step 4: Launch FastAPI Server & Expose via Cloudflare Tunnel
import subprocess
import threading
import time
import re
import uvicorn

# Start Uvicorn in background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print("[+] Uvicorn server listening on port 8000.")

# Launch Cloudflared tunnel
print("[*] Launching secure Cloudflare public tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!")
    print(f"👉 COPY THIS URL INTO YOUR WEB APP:\n{tunnel_url}")
    print("="*60 + "\n")
else:
    print("[!] Cloudflare tunnel did not output URL yet. Check output above.")